# 05 — Model: CatBoost

Requires `train_features.csv` / `test_features.csv` from **01_feature_engineering.ipynb**.

Run `!pip install catboost` once if you don't already have it.

CatBoost's whole design point is native categorical handling — pass the raw string columns (no encoding
needed) and list them as `cat_features`. Often the strongest single model on datasets with categorical
columns, though here the categoricals themselves are weak, so the gain over LightGBM/XGBoost may be small.
Saves `oof_cat.csv` and `test_pred_cat.csv` for the ensembling notebook.

In [1]:
!pip install -q catboost


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv / test_features.csv from notebook 01
N_FOLDS = 5
SEED = 42

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
test_fe = pd.read_csv(f"{DATA_DIR}/test_features.csv")

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

X = train_fe[feat_cols].copy()
Xtest = test_fe[feat_cols].copy()
y = train_fe['addicted_label'].values

for c in cat_cols:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(X.shape, Xtest.shape)

from catboost import CatBoostClassifier
import time

# CatBoost wants NaN filled for categorical columns
X_cb = X.copy()
Xtest_cb = Xtest.copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(str).replace('nan', 'missing')
    Xtest_cb[c] = Xtest_cb[c].astype(str).replace('nan', 'missing')


(691369, 30) (296302, 30)


In [3]:
oof_cat = np.zeros(len(X))
test_cat = np.zeros(len(Xtest))

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    model = CatBoostClassifier(
        iterations=3000, learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
        eval_metric='AUC', loss_function='Logloss', random_seed=SEED,
        cat_features=cat_cols, early_stopping_rounds=100, verbose=False
    )
    model.fit(
        X_cb.iloc[tr_idx], y[tr_idx],
        eval_set=(X_cb.iloc[va_idx], y[va_idx])
    )
    p_va = model.predict_proba(X_cb.iloc[va_idx])[:, 1]
    oof_cat[va_idx] = p_va
    test_cat += model.predict_proba(Xtest_cb)[:, 1] / N_FOLDS
    print(f"fold {fold} auc={roc_auc_score(y[va_idx], p_va):.5f}  best_iter={model.get_best_iteration()}  ({time.time()-t0:.0f}s elapsed)")

print("CatBoost OOF AUC:", roc_auc_score(y, oof_cat))


fold 0 auc=0.96169  best_iter=2999  (925s elapsed)
fold 1 auc=0.96207  best_iter=2999  (1853s elapsed)
fold 2 auc=0.96230  best_iter=2999  (2786s elapsed)
fold 3 auc=0.96317  best_iter=2999  (3712s elapsed)
fold 4 auc=0.96220  best_iter=2999  (4634s elapsed)
CatBoost OOF AUC: 0.962282490698824


In [4]:
pd.DataFrame({'id': train_fe['id'], 'oof_pred': oof_cat}).to_csv(f"{DATA_DIR}/oof_cat.csv", index=False)
pd.DataFrame({'id': test_fe['id'], 'test_pred': test_cat}).to_csv(f"{DATA_DIR}/test_pred_cat.csv", index=False)
print("saved oof_cat.csv and test_pred_cat.csv")


saved oof_cat.csv and test_pred_cat.csv
